In [43]:
import requests
import re
import json
from typing import List, Dict

# 你可以自己加/删
PAGE_TITLES = [
    "Japanese cuisine",
    "Chinese cuisine",
    "Korean cuisine",
    "Sushi",
    "Kimchi",
    "Ramen",
    "Soy sauce",
    "Tofu",
    "Tempura",
    "Udon",
    "Soba",
    "Peking duck",
    "Hot pot",
    "Dim sum",
    "Bibimbap",
    "Bulgogi",
    "Tteokbokki",
    "Rice",
    "Noodle",
    "Miso",
    "Seaweed",
    "Fish sauce",
    "Ginger",
    "Garlic",
    "Stir frying",
    "Steaming",
    "Deep frying",
    "Fermentation",
    "Braising",
    "Japanese tea ceremony",
    "Korean barbecue",
    "Street food"
]

In [44]:
def fetch_wikipedia_page(title: str):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": title,
        "format": "json",
        "redirects": 1
    }

    headers = {
        "User-Agent": "UoM-TRIM-coursework-benchmark/1.0 (student project)"
    }

    r = requests.get(url, params=params, headers=headers, timeout=20)
    r.raise_for_status()
    data = r.json()

    pages = data["query"]["pages"]
    page = next(iter(pages.values()))

    return {
        "title": title,
        "text": page.get("extract", "")
    }

In [45]:
def clean_text(text: str) -> str:
    text = text.replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def split_sentences(text: str) -> List[str]:
    text = clean_text(text)
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

In [46]:
def get_definition_sentence(title: str, text: str):
    sentences = split_sentences(text[:2500])
    title_lower = title.lower()

    for s in sentences:
        s_clean = clean_text(s)
        if s_clean.lower().startswith(title_lower):
            return s_clean

    for s in sentences:
        s_clean = clean_text(s)
        if title_lower in s_clean.lower():
            return s_clean

    return None

In [47]:
def build_benchmark(page_titles: List[str]) -> List[Dict]:
    benchmark = []
    seen_queries = set()

    # 只保留最稳的页面
    allowed_origin_pages = {
        "tempura",
        "peking duck"
    }

    allowed_served_pages = {
        "peking duck",
        "hot pot"
    }

    allowed_when_pages = {
        "udon",
        "dim sum"
    }

    allowed_used_for_pages = {
        "soy sauce",
        "miso",
        "garlic"
    }

    origin_patterns = [
        "originated in",
        "originated from"
    ]

    served_patterns = [
        "served with",
        "served in",
        "served on",
        "served inside"
    ]

    when_patterns = [
        "traditionally enjoyed",
        "for brunch",
        "at nearly every meal",
        "usually served",
        "usually eaten",
        "commonly served",
        "commonly eaten"
    ]

    used_for_patterns = [
        "used for",
        "used in",
        "used as",
        "is used for",
        "is used in",
        "is used as",
        "commonly used as",
        "commonly used in",
        "most commonly appears as",
        "appears as"
    ]

    bad_substrings = [
        "lit.",
        "etymology",
        "history",
        "variation of",
        "kitakata ramen",
        "edomae chirashizushi",
        "edo period",
        "roman times"
    ]

    def clean_answer_text(answer: str) -> str:
        answer = clean_text(answer)
        answer = re.sub(r"^==+\s*[^=]+?\s*==+\s*", "", answer).strip()
        answer = re.sub(r"^=+\s*[^=]+?\s*=+\s*", "", answer).strip()
        return answer

    def is_good_answer(answer: str) -> bool:
        if not answer:
            return False
        a = answer.lower()
        if len(answer) < 20 or len(answer) > 220:
            return False
        if any(bad in a for bad in bad_substrings):
            return False
        return True

    def add_example(query, answer):
        if not answer:
            return
        answer = clean_answer_text(answer)
        if query not in seen_queries and is_good_answer(answer):
            benchmark.append({
                "query": query,
                "answer": answer
            })
            seen_queries.add(query)

    def first_match(sentences, title_lower, patterns, limit=None):
        sents = sentences if limit is None else sentences[:limit]
        for s in sents:
            s_clean = clean_answer_text(s)
            s_lower = s_clean.lower()
            if title_lower in s_lower and any(p in s_lower for p in patterns):
                if is_good_answer(s_clean):
                    return s_clean
        return None

    for title in page_titles:
        try:
            doc = fetch_wikipedia_page(title)
            text = doc["text"]
        except Exception as e:
            print(f"Skip {title}: {e}")
            continue

        if not text:
            continue

        sentences = split_sentences(text)
        if not sentences:
            continue

        title_lower = title.strip().lower()

        # 1) Where does X come from?
        if title_lower in allowed_origin_pages:
            ans = first_match(sentences, title_lower, origin_patterns, limit=20)
            add_example(f"Where does {title_lower} come from?", ans)

        # 2) How is X usually served?
        if title_lower in allowed_served_pages:
            ans = first_match(sentences, title_lower, served_patterns)
            add_example(f"How is {title_lower} usually served?", ans)

        # 3) When is X usually eaten or served?
        if title_lower in allowed_when_pages:
            ans = first_match(sentences, title_lower, when_patterns)
            add_example(f"When is {title_lower} usually eaten or served?", ans)

        # 4) What is X used for?
        if title_lower in allowed_used_for_pages:
            ans = first_match(sentences, title_lower, used_for_patterns)
            add_example(f"What is {title_lower} used for?", ans)

    return benchmark

In [48]:
benchmark = build_benchmark(PAGE_TITLES)

print("Number of examples:", len(benchmark))
print(json.dumps(benchmark, indent=2, ensure_ascii=False))

with open("benchmark.json", "w", encoding="utf-8") as f:
    json.dump(benchmark, f, indent=2, ensure_ascii=False)

print("Saved benchmark.json")

Number of examples: 8
[
  {
    "query": "What is soy sauce used for?",
    "answer": "Soy sauce may be added directly to food and is commonly used as a dipping sauce or used as seasoning in cooking."
  },
  {
    "query": "Where does tempura come from?",
    "answer": "Tempura originated in the 16th century, when Portuguese Jesuits brought the Western-style cooking method of coating foods with flour and frying, via Nanban trade."
  },
  {
    "query": "When is udon usually eaten or served?",
    "answer": "Udon noodles are usually served chilled in the summer and hot in the winter."
  },
  {
    "query": "How is peking duck usually served?",
    "answer": "Peking duck is traditionally carved in front of the guests and served in three stages."
  },
  {
    "query": "How is hot pot usually served?",
    "answer": "Hot pot is a flavorful broth traditionally served inside a large metal pot."
  },
  {
    "query": "When is dim sum usually eaten or served?",
    "answer": "Dim sum (traditio